In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random
import uuid

spark.sql("CREATE DATABASE IF NOT EXISTS parallax_demo")
spark.sql("USE parallax_demo")

In [0]:
random.seed(42)

markets = [
    "tesla_model_z_before_june_30",
    "fed_rate_cut_next_meeting",
    "btc_above_100k_year_end",
    "election_candidate_x_wins",
]

start = datetime(2026, 4, 28, 9, 0, 0)
rows = []

for market in markets:
    price = random.uniform(0.2, 0.6)

    for i in range(500):
        ts = start + timedelta(seconds=i * random.randint(20, 90))

        # normal random walk
        price += random.uniform(-0.01, 0.01)
        price = max(0.01, min(0.99, price))

        size = random.uniform(10, 300)
        side = random.choice(["YES", "NO"])

        # Inject suspicious pattern into Tesla market
        if market == "tesla_model_z_before_june_30" and 250 <= i <= 290:
            side = "YES"
            size = random.uniform(1000, 8000)
            price += random.uniform(0.005, 0.02)
            price = min(0.99, price)

        rows.append({
            "event_id": str(uuid.uuid4()),
            "venue": "synthetic",
            "market_id": market,
            "contract_id": market + "_yes",
            "ts": ts.isoformat(),
            "side": side,
            "price": float(round(price, 4)),
            "size": float(round(size, 2))
        })

bronze_df = spark.createDataFrame(rows)

bronze_df.write.mode("overwrite").format("delta").saveAsTable("bronze_market_events")

display(bronze_df.limit(10))

In [0]:
bronze = spark.table("bronze_market_events")

silver = (
    bronze
    .withColumn("ts", F.to_timestamp("ts"))
    .withColumn("window", F.window("ts", "5 minutes"))
    .groupBy(
        "market_id",
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end")
    )
    .agg(
        F.count("*").alias("trade_count"),
        F.sum("size").alias("volume"),
        F.avg("price").alias("avg_price"),
        F.min("price").alias("low_price"),
        F.max("price").alias("high_price"),
        F.sum(F.when(F.col("side") == "YES", F.col("size")).otherwise(0)).alias("yes_volume"),
        F.sum(F.when(F.col("side") == "NO", F.col("size")).otherwise(0)).alias("no_volume")
    )
    .withColumn("imbalance", (F.col("yes_volume") - F.col("no_volume")) / F.col("volume"))
    .withColumn("price_range", F.col("high_price") - F.col("low_price"))
    .withColumn("price_impact", F.col("price_range") / F.col("volume"))
)

silver.write.mode("overwrite").format("delta").saveAsTable("silver_market_features")

display(silver.orderBy(F.desc("volume")).limit(20))

In [0]:
silver = spark.table("silver_market_features")

market_stats = (
    silver
    .groupBy("market_id")
    .agg(
        F.avg("volume").alias("avg_volume"),
        F.stddev("volume").alias("std_volume"),
        F.avg("trade_count").alias("avg_trade_count"),
        F.stddev("trade_count").alias("std_trade_count")
    )
)

scored = (
    silver
    .join(market_stats, on="market_id", how="left")
    .withColumn(
        "volume_z",
        (F.col("volume") - F.col("avg_volume")) / F.when(F.col("std_volume") == 0, 1).otherwise(F.col("std_volume"))
    )
    .withColumn(
        "trade_count_z",
        (F.col("trade_count") - F.col("avg_trade_count")) / F.when(F.col("std_trade_count") == 0, 1).otherwise(F.col("std_trade_count"))
    )
    .withColumn("volume_score", F.least(F.greatest(F.col("volume_z") * 20, F.lit(0)), F.lit(40)))
    .withColumn("clustering_score", F.least(F.greatest(F.col("trade_count_z") * 15, F.lit(0)), F.lit(25)))
    .withColumn("imbalance_score", F.least(F.abs(F.col("imbalance")) * 25, F.lit(25)))
    .withColumn("price_range_score", F.least(F.col("price_range") * 200, F.lit(10)))
    .withColumn(
        "anomaly_score",
        F.round(
            F.col("volume_score") +
            F.col("clustering_score") +
            F.col("imbalance_score") +
            F.col("price_range_score"),
            2
        )
    )
    .withColumn(
        "reason_tags",
        F.array_remove(F.array(
            F.when(F.col("volume_z") > 2, F.lit("volume_burst")),
            F.when(F.col("trade_count_z") > 2, F.lit("trade_clustering")),
            F.when(F.abs(F.col("imbalance")) > 0.7, F.lit("one_sided_flow")),
            F.when(F.col("price_range") > 0.05, F.lit("price_dislocation"))
        ), None)
    )
)

gold = scored.select(
    "market_id",
    "window_start",
    "window_end",
    "volume",
    "trade_count",
    "imbalance",
    "price_range",
    "volume_z",
    "trade_count_z",
    "anomaly_score",
    "reason_tags"
)

gold.write.mode("overwrite").format("delta").saveAsTable("gold_parallax_signals")

display(gold.orderBy(F.desc("anomaly_score")).limit(20))

In [0]:
gold = spark.table("gold_parallax_signals")

display(
    gold
    .withColumn(
        "behavior_label",
        F.when(
            (F.col("anomaly_score") > 70) & (F.abs(F.col("imbalance")) > 0.75),
            F.lit("momentum_push_or_aggressive_accumulation")
        ).when(
            (F.col("anomaly_score") > 40) & (F.col("volume_z") > 2),
            F.lit("unusual_accumulation")
        ).otherwise(F.lit("normal_or_low_signal"))
    )
    .orderBy(F.desc("anomaly_score"))
)